# 00 Planung und Probebilder

**Vor** der Aufnahmereihe: Ziel und Methode festlegen, wenige **Probebilder** auswerten, daraus Belichtung, Fokus/Defokus und Aperturen ableiten — und als **Plan** speichern, den die Folge-Notebooks lesen können.

**Warum:** Eine Aufnahmereihe lässt sich nicht nachträglich retten, wenn das Ziel gesättigt ist, die Sterne bei einer Bayer-Kamera zu scharf sind oder die Belichtung außerhalb des linearen Bereichs liegt.
Das AAVSO-DSLR-Handbuch (v1.4, gelesen) verlangt: alle Sterne (Ziel, Comps, Check) unter **75 % des Maximalwerts** (§4.5.2), **FWHM im Rohbild ≥ 8–10 px** bei Bayer-Kameras (§5.5.1) und einen Aperturradius von
**1,2–1,5 × FWHM** (§6.1.1). Diese Werte kennt man erst aus Probebildern.

| Schritt | Was | Ergebnis |
|---|---|---|
| 1 | Ziel, Methode (Monochrom + Filter / Farbkamera / DSLR), Kamera-Profil (NB07) | Meldefilter (`V` oder `TG`/`TB`/`TR`), `OBSTYPE` |
| 2 | Standort (NB01), Sichtbarkeit in der Nacht | Zeitfenster (dunkel, Ziel > 30°) |
| 3 | **Probebilder** auswerten: hellster Stern, Sättigung, FWHM | Messwerte je Bild |
| 4 | Empfehlung: längste/empfohlene Belichtung, Aperturen, Warnungen | Plan |
| 5 | Plan speichern | `PHOTOMETRIE_PY_DATA/plans/*.json` |

**Probebilder aufnehmen:** 3–5 Aufnahmen **desselben Feldes** mit dem Aufbau der späteren Reihe (Fokus/Defokus wie geplant), unterschiedlich belichtet (z. B. 0,5×, 1×, 2× der geschätzten Zeit), Ziel im Bild.
Das Ziel in seiner **hellsten Phase** planen (Veränderliche werden heller!).

## Schritt 1 — Ziel, Methode und Kamera

`MODE="demo"`: synthetische Probebilder (zeigt die Auswertung und die Warnungen). `MODE="real"`: eigene Bilder aus `TEST_DIR`. Vorbelegen per Umgebungsvariable (`VARPHOT_TARGET`, `VARPHOT_METHOD`, `VARPHOT_NB00_MODE`).

In [ ]:
import os, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

from varphot import (METHODS, ObservingPlan, PRESETS, aperture_from_fwhm, fwhm_summary, green_plane, night_altitudes, recommend_exposure,
                     save_plan, star_metrics, get_preset)
from varphot.plan import check_field_stars, comp_v_min, plane_wcs
from varphot.observer import all_sites, camera_label, data_dir, load_config
from varphot.plan import SAFE_FRACTION, MIN_RAW_FWHM_BAYER
from varphot.testdata import testdata_dir
from varphot.vsp import resolve_target
from varphot.batch import read_frame

MODE = os.environ.get("VARPHOT_NB00_MODE", "demo")                  # "demo" oder "real"
_obs, _own = load_config()
_cams = list(_obs.cameras)
if _obs.camera.model and all((c.model, c.gain_setting) != (_obs.camera.model, _obs.camera.gain_setting) for c in _cams):
    _cams.append(_obs.camera)
_cam_opts = [(f"Profil: {camera_label(c)}", ("profil", i)) for i, c in enumerate(_cams)] + [(f"Vorlage: {p.model}", ("vorlage", k)) for k, p in PRESETS.items() if k != "eigene"]
_st = {"description_width": "110px"}; _lay = widgets.Layout(width="560px")
w_target = widgets.Text(description="Ziel:", value=os.environ.get("VARPHOT_TARGET", "del Cep"), style=_st, layout=_lay)
w_vmax = widgets.Text(description="V (hellste Phase):", value=os.environ.get("VARPHOT_VMAX", ""), placeholder="leer = SIMBAD-Wert", style=_st, layout=_lay)
w_method = widgets.Dropdown(description="Methode:", options=[(v, k) for k, v in METHODS.items()], value=os.environ.get("VARPHOT_METHOD", "osc"), style=_st, layout=_lay)
w_cam = widgets.Dropdown(description="Kamera:", options=_cam_opts, style=_st, layout=_lay)
w_filter = widgets.Dropdown(description="Meldefilter:", options=["V", "B", "R", "TG", "TB", "TR"], value="TG", style=_st, layout=_lay)
def _meth(*_):
    w_filter.options = ["V", "B", "R"] if w_method.value == "mono" else ["TG", "TB", "TR", "V", "B", "R"]
    w_filter.value = "V" if w_method.value == "mono" else "TG"
w_method.observe(_meth, names="value"); _meth()
display(widgets.VBox([w_target, w_vmax, w_method, w_cam, w_filter]))

### Auswahl übernehmen

Löst das Ziel über SIMBAD auf (bei fehlendem Netz: `RA_DEG`/`DEC_DEG`/`V_MEAN` unten von Hand setzen) und liest das Kamera-Profil.

In [ ]:
TARGET, METHOD, AAVSO_FILTER = w_target.value.strip(), w_method.value, w_filter.value
kind, ref = w_cam.value
if kind == "profil":
    CAM, PRESET = _cams[ref], None
else:
    PRESET = get_preset(ref); CAM = None
MODEL = CAM.model if CAM else PRESET.model
SETTING = CAM.gain_setting if CAM else next(iter(PRESET.settings))
PATTERN = (CAM.bayer_pattern if CAM else PRESET.bayer_pattern) or "RGGB"
OBSTYPE = "CCD" if METHOD == "mono" else "DSLR"                       # vorläufig: auch Mono + RGB-Filter zählt in der AAVSO-Forum-Zusammenfassung als "DSLR" (KONZEPT §11.2/§11.12)
try:
    T = resolve_target(TARGET)
    RA_DEG, DEC_DEG, V_MEAN = T["ra"], T["dec"], T.get("V")
except Exception as e:                                                  # offline: von Hand setzen
    print("SIMBAD nicht erreichbar:", e)
    RA_DEG, DEC_DEG, V_MEAN = 337.2928, 58.4152, 3.75                    # <- bei Bedarf eintragen (hier δ Cep)
V_MAX = float(w_vmax.value.replace(",", ".")) if w_vmax.value.strip() else V_MEAN
BITS = (CAM.bit_depth if CAM and CAM.bit_depth else None) or (PRESET.bit_depth if PRESET else None) or 14
OFFSET = CAM.offset_adu if CAM and CAM.offset_adu is not None else None
MAX_ADU = CAM.saturation_adu if CAM and CAM.saturation_adu else (2 ** BITS - 1)
if CAM and CAM.linear_max_adu:
    LIMIT, LIMIT_SRC = float(CAM.linear_max_adu), "gemessene Linearitätsgrenze (NB07)"
else:
    LIMIT, LIMIT_SRC = SAFE_FRACTION * MAX_ADU, f"{int(SAFE_FRACTION * 100)} % des Maximalwerts (Handbuch §4.5.2; kein NB07-Profil)"
print(f"Ziel {TARGET}: RA {RA_DEG:.4f}°, Dec {DEC_DEG:+.4f}°, V = {V_MEAN} (hellste Phase für die Planung: {V_MAX})")
print(f"Methode: {METHODS[METHOD]} → Meldefilter {AAVSO_FILTER}, OBSTYPE {OBSTYPE}")
print(f"Kamera: {MODEL} — {SETTING} ({'Profil' if CAM else 'Vorlage ohne Messung'}), Bayer {PATTERN if METHOD in ('osc', 'dslr') else '—'}, Maximalwert {MAX_ADU:.0f} ADU, "
      f"Obergrenze für Sterne {LIMIT:.0f} ADU [{LIMIT_SRC}]")
if CAM is None:
    print("HINWEIS: für diese Kamera gibt es noch kein NB07-Profil — Offset, Maximalwert und Linearität sind nominell.")

## Schritt 2 — Standort und Sichtbarkeit

Standort aus NB01. Grün = dunkel (Sonne < −12°) und Ziel über 30°. Der **Mond** wird mitgerechnet (Höhe, Abstand zum Ziel, beleuchteter Anteil): er hellt den Himmel auf (Himmelsrauschen, Gradienten im Himmelsring) — bei hellen Zielen wie δ Cep meist unkritisch, bei schwachen schon. "Mond ok" heißt: Mond unter dem Horizont oder Abstand > 40° bei Beleuchtung < 50 % (Startwerte). Ohne Wetter.

In [ ]:
SITE = next((s for s in all_sites(_own) if s.name == _obs.site_name), None)
MOON = {}
if SITE is None:
    print("Kein Standort gespeichert — NB01 Abschnitt 0 ausfüllen. Sichtbarkeit übersprungen.")
else:
    NA = night_altitudes(RA_DEG, DEC_DEG, SITE, step_min=30)
    ok = NA[NA["ok"]]
    print(f"Standort {SITE.name}: " + (f"Beobachtungsfenster (UTC) {ok['utc'].iloc[0]} bis {ok['utc'].iloc[-1]}, Luftmasse {ok['airmass'].min():.2f}–{ok['airmass'].max():.2f}" if len(ok) else "heute Nacht kein Fenster (dunkel und > 30°)"))
    MOON = {}
    if len(ok):
        okm = NA[NA["ok_moon"]]
        MOON = dict(illum=round(float(ok["moon_illum"].mean()), 2), sep_deg=round(float(ok["moon_sep"].mean())), up_fraction=round(float((ok["alt_moon"] > 0).mean()), 2),
                    window_without_moon_disturbance=int(len(okm)) * 30)
        print(f"Mond im Fenster: Beleuchtung {100 * MOON['illum']:.0f} %, mittlerer Abstand zum Ziel {MOON['sep_deg']}°, {100 * MOON['up_fraction']:.0f} % der Zeit über dem Horizont; "
              f"ungestörtes Fenster {MOON['window_without_moon_disturbance']} min")
        if len(okm) < len(ok):
            print("⚠ Mond stört in Teilen des Fensters" + (" (heller Mond nahe am Ziel)" if MOON["illum"] > 0.5 and MOON["sep_deg"] < 40 else ""))
    fig, ax = plt.subplots(figsize=(8, 2.8))
    ax.plot(range(len(NA)), NA["alt_target"], label="Ziel"); ax.plot(range(len(NA)), NA["alt_sun"], color="orange", label="Sonne"); ax.plot(range(len(NA)), NA["alt_moon"], color="gray", ls="--", label="Mond")
    ax.fill_between(range(len(NA)), -90, 90, where=NA["ok_moon"], color="g", alpha=0.18, label="Fenster (Mond ok)"); ax.fill_between(range(len(NA)), -90, 90, where=NA["ok"] & ~NA["ok_moon"], color="y", alpha=0.18, label="Fenster (Mond stört)"); ax.axhline(30, color="k", ls=":", lw=0.6)
    ax.set_xticks(range(0, len(NA), 4)); ax.set_xticklabels([u[11:] for u in NA["utc"][::4]], rotation=45); ax.set_ylim(-60, 90); ax.set_ylabel("Höhe [°]"); ax.legend(loc="upper right")
    plt.tight_layout(); plt.show()

## Schritt 3 — Probebilder auswerten

Je Bild: hellster Stern (Spitzenwert mit Offset, gesättigt?) und **FWHM** der mittelhellen, ungesättigten Sterne (Gauß-Anpassung). Bei Bayer-Kameras wird auf der **Grünebene** gemessen (beide Grün gemittelt, halbe Auflösung);
die Rohbild-FWHM ist doppelt so groß und muss ≥ 8–10 px sein. Im Modus `real` stehen die Probebilder als FITS (bei RAW: zuerst mit Siril ohne Debayern wandeln, siehe NB07) in `TEST_DIR`.

In [ ]:
TEST_DIR = Path(os.environ.get("VARPHOT_TEST_DIR") or testdata_dir() / "plan" / f"{TARGET.replace(' ', '_')}")
BAYER = METHOD in ("osc", "dslr")                                  # nur Bayer-Kameras: Grünebene, FWHM im Rohbild ≥ 8–10 px

STARS_DEMO = [(400, 500, 3800.0), (150, 200, 900.0), (600, 300, 600.0), (250, 800, 450.0), (650, 850, 300.0), (120, 650, 220.0), (700, 120, 160.0)]   # (y, x, Spitzenwert ADU/s); erster = Ziel
def synth_frame(fwhm_raw, exptime, seed):
    """Syntetisches Probebild: Sternfeld, Bayer-Empfindlichkeiten (R 0,8 / G 1,0 / B 0,7), Ziel (der hellste Stern) wird mit der Belichtung heller."""
    rng = np.random.default_rng(seed)
    ny, nx = 800, 1000
    yy, xx = np.mgrid[0:ny, 0:nx]
    img = 100.0 + rng.normal(0, 4.0, (ny, nx))
    s = fwhm_raw / 2.355
    for y0, x0, rate in STARS_DEMO:                                              # rate = Spitzenwert in ADU je Sekunde
        img += rate * exptime * np.exp(-((xx - x0) ** 2 + (yy - y0) ** 2) / (2 * s * s))
    if BAYER:
        q = np.ones((ny, nx)); q[0::2, 0::2] = 0.8; q[1::2, 1::2] = 0.7
        img = 100.0 + (img - 100.0) * q
    return np.clip(np.rint(img), 0, MAX_ADU)

frames = []
if MODE == "demo":
    FW_DEMO = 6.0 if BAYER else 4.5                                         # Bayer-Demo absichtlich zu scharf → Warnung
    for k, t in enumerate((0.5, 1.0, 2.0, 4.0)):
        frames.append((f"synthetisch_{t:g}s", t, synth_frame(FW_DEMO, t, 10 + k)))
    print(f"DEMO: {len(frames)} synthetische Probebilder (Rohbild-FWHM {FW_DEMO} px)")
else:
    from varphot.camera import exposure_from_header
    for p in sorted(TEST_DIR.glob("*.fit*")):
        img, hdr, _ = read_frame(p)
        t = exposure_from_header(hdr)
        if t is None:
            raise RuntimeError(f"{p.name}: keine Belichtungszeit in der Kopfzeile (EXPTIME/EXPOSURE)")
        frames.append((p.name, t, img))
    print(f"{len(frames)} Probebilder aus {TEST_DIR}")

rows, FW_ALL = [], []
for name, t, img in frames:
    work = green_plane(img, PATTERN) if BAYER else np.asarray(img, float)
    lim = MAX_ADU
    df = star_metrics(work, max_adu=lim, fwhm_guess=3.0 if BAYER else 4.0)
    fw = fwhm_summary(df)
    top = df.iloc[0] if len(df) else None
    off = OFFSET if OFFSET is not None else float(np.median(work))
    rows.append(dict(Bild=name, Belichtung_s=t, hellster_Stern_ADU=None if top is None else round(top["raw_max"]), gesättigt=None if top is None else bool(top["saturated"]),
                     Anteil_am_Maximum=None if top is None else round(top["raw_max"] / MAX_ADU, 2), FWHM_px=round(fw["fwhm"], 2), n_Sterne=fw["n"], Offset=round(off)))
    if fw["n"]:
        FW_ALL.append(fw["fwhm"])
TAB = pd.DataFrame(rows)
display(TAB)

## Schritt 3b — Jeden Katalogstern prüfen (Ziel, Comps, Check)

Nicht nur der hellste Stern des Bildes zählt, sondern **jeder Stern, der in die Photometrie eingeht**: keiner darf über die Obergrenze (Handbuch §4.5.2), und jeder Comp braucht genug Signal (**S/N ≥ 100**, §6.3.1). Das Ziel wird für seine hellste Phase gerechnet.
Dazu braucht es (1) die Positionen der Sterne im Probebild (WCS aus der Kopfzeile oder per ASTAP, bei Bayer-Bildern auf die Grünebene umgerechnet) und (2) einen Katalog der Sterne (`field_config.json` aus NB03, `VARPHOT_FIELD_CONFIG`).
Ergebnis: je Stern die längste zulässige Belichtung, das S/N und die **hellste zulässige Comp-Helligkeit** `V_comp_min` (für NB03).

In [ ]:
FIELD, T_MAX_FIELD, V_COMP_MIN = None, None, None
TARGET_IS_BRIGHTEST = True            # das Ziel ist der hellste Stern im Feld (δ Cep: ja)
ref = None
FWHM_TMP = float(np.median(FW_ALL)) if FW_ALL else 4.0
okf = TAB[(TAB["gesättigt"] == False) & TAB["hellster_Stern_ADU"].notna()]
if len(okf):
    ref = okf.sort_values("hellster_Stern_ADU", ascending=False).iloc[0]            # ungesättigtes Bild mit dem besten Signal
if ref is None:
    print("kein ungesättigtes Probebild — Sternprüfung übersprungen")
else:
    name_ref, t_ref, img_ref = next(f for f in frames if f[0] == ref["Bild"])
    if MODE == "demo":
        from astropy.wcs import WCS
        w_raw = WCS(naxis=2); w_raw.wcs.crpix = [500, 400]; w_raw.wcs.crval = [RA_DEG, DEC_DEG]; w_raw.wcs.ctype = ["RA---TAN", "DEC--TAN"]
        w_raw.wcs.cdelt = [-1.44 / 3600, 1.44 / 3600]
        rows = []
        for k, (y0, x0, rate) in enumerate(STARS_DEMO):
            ra, dec = w_raw.pixel_to_world_values(x0, y0)
            rows.append(dict(id="Ziel" if k == 0 else f"C{k}", role="target" if k == 0 else "comp", ra=float(ra), dec=float(dec), V=V_MEAN - 2.5 * np.log10(rate / STARS_DEMO[0][2])))
        CATSTARS = pd.DataFrame(rows)
    else:
        import json
        from varphot.vsp import vsp_table  # noqa: F401
        fc_path = Path(os.environ.get("VARPHOT_FIELD_CONFIG", f"../data/fields/{TARGET.replace(' ', '_')}_field_config.json"))
        if not fc_path.exists():
            CATSTARS, w_raw = None, None
            print("Keine field_config.json gefunden:", fc_path, "— NB03 ausführen oder VARPHOT_FIELD_CONFIG setzen; Sternprüfung übersprungen.")
        else:
            rec = json.loads(fc_path.read_text())
            rows = [dict(id="Ziel", role="target", ra=rec["ziel"]["ra"], dec=rec["ziel"]["dec"], V=V_MEAN)]
            rows += [dict(id=str(s["label"]), role=s["rolle"], ra=s["ra"], dec=s["dec"], V=s["V"]) for s in rec["sterne"] if s["rolle"] in ("comp", "check")]
            CATSTARS = pd.DataFrame(rows)
            _img, _hdr, w_raw = read_frame(TEST_DIR / name_ref)
            if w_raw is None:
                from varphot.astrometry import solve_image
                sr = solve_image(TEST_DIR / name_ref)
                w_raw = sr.wcs
                print("WCS per ASTAP gelöst:", sr.source)
    if CATSTARS is not None and w_raw is not None:
        work = green_plane(img_ref, PATTERN) if BAYER else np.asarray(img_ref, float)
        w_use = plane_wcs(w_raw, (0.5, 0.5)) if BAYER else w_raw
        offv = OFFSET if OFFSET is not None else float(np.median(work))
        gain_v = (CAM.gain_e_per_adu if CAM and CAM.gain_e_per_adu else 1.0); rn_v = (CAM.readout_noise_e if CAM and CAM.readout_noise_e else 0.0)
        if not (CAM and CAM.gain_e_per_adu):
            print("HINWEIS: kein Gain/Rauschen aus NB07 — S/N-Angaben sind nur grob (Gain 1, kein Ausleserauschen).")
        T_REC_TMP = float(ref["Belichtung_s"])
        FIELD = check_field_stars(work, w_use, CATSTARS, exptime_s=float(ref["Belichtung_s"]), limit_adu=LIMIT, offset=offv, aperture=aperture_from_fwhm(FWHM_TMP),
                                  gain=gain_v, read_noise_e=rn_v, max_adu=MAX_ADU, snr_min=100.0)
        if TARGET_IS_BRIGHTEST:
            f_phase = 10 ** (0.4 * (V_MEAN - V_MAX)) if (V_MAX is not None and V_MEAN is not None and V_MAX < V_MEAN) else 1.0
            FIELD.loc[FIELD["role"] == "target", "t_max_s"] /= f_phase                 # Ziel in der hellsten Phase
        T_MAX_FIELD = float(FIELD["t_max_s"].min()) if FIELD["t_max_s"].notna().any() else None
        FIELD["limiting"] = FIELD["t_max_s"] == FIELD["t_max_s"].min()
        show = FIELD[["id", "role", "V", "raw_max", "over_limit", "t_max_s", "snr", "ok_snr", "limiting", "star_flag"]].rename(columns={"raw_max": "hellster_Pixel_ADU", "over_limit": "über_Grenze", "t_max_s": "t_max_s", "snr": "S/N (Probebild)", "ok_snr": "S/N≥100"})
        display(show.round(2))
        print(f"Begrenzender Stern: {FIELD.loc[FIELD['limiting'], 'id'].iloc[0]} ({FIELD.loc[FIELD['limiting'], 'role'].iloc[0]}) → längste Belichtung {T_MAX_FIELD:.2f} s")

## Schritt 4 — Empfehlung

* **Belichtung:** Aus jedem **ungesättigten** Probebild folgt die längste Zeit, bei der der hellste Stern unter der Obergrenze bleibt (lineare Skalierung); es gilt das Minimum. Wird das Ziel bei hellster Phase heller als im Probebild,
  wird entsprechend gekürzt (`TARGET_IS_BRIGHTEST`: das Ziel ist der hellste Stern im Feld). Empfohlen werden 80 % der längsten Zeit.
* **Aperturen:** 1,5 × FWHM der Ebene (Handbuch §6.1.1), Himmelsring nach §6.1.2.
* **Warnungen:** Bayer-FWHM im Rohbild < 8 px (zu scharf → defokussieren), alle Probebilder gesättigt, FWHM sehr groß.

In [ ]:
TARGET_IS_BRIGHTEST = True
WARN = []
FWHM_PX = float(np.median(FW_ALL)) if FW_ALL else float("nan")
FWHM_RAW = FWHM_PX * (2 if BAYER else 1)
usable = TAB[(TAB["gesättigt"] == False) & TAB["hellster_Stern_ADU"].notna()]
if usable.empty:
    T_MAX = None
    WARN.append("Alle Probebilder sind gesättigt (oder ohne Sterne): mit kürzerer Belichtung bzw. mehr Defokus neu aufnehmen.")
else:
    t_list = [recommend_exposure(r["hellster_Stern_ADU"], r["Belichtung_s"], LIMIT, r["Offset"]) for _, r in usable.iterrows()]
    T_MAX = float(min(t_list))
    if TARGET_IS_BRIGHTEST and V_MAX is not None and V_MEAN is not None and V_MAX < V_MEAN:
        T_MAX /= 10 ** (0.4 * (V_MEAN - V_MAX))
        print(f"Korrektur auf die hellste Phase (V {V_MEAN} → {V_MAX}): Belichtung × {10 ** (-0.4 * (V_MEAN - V_MAX)):.2f}")
if T_MAX_FIELD is not None:
    T_MAX = T_MAX_FIELD if T_MAX is None else min(T_MAX, T_MAX_FIELD)                # jeder Katalogstern zählt, nicht nur der hellste Stern des Bildes
T_REC = None if T_MAX is None else round(0.8 * T_MAX, 1 if T_MAX < 30 else 0)
if FIELD is not None and T_REC is not None:
    FIELD["ok_snr_rec"] = FIELD["snr"] * np.sqrt(T_REC / float(ref["Belichtung_s"])) >= 100
    V_COMP_MIN = comp_v_min(FIELD[FIELD["role"] == "comp"], float(ref["Belichtung_s"]), T_REC, LIMIT, offv)
    comps = FIELD[FIELD["role"] == "comp"]
    over = comps[comps["t_max_s"] < T_REC]
    faint = comps[~comps["ok_snr_rec"]]
    if len(over):
        WARN.append("Comps über der Obergrenze bei der empfohlenen Belichtung: " + ", ".join(over["id"]) + " (nicht verwenden oder kürzer belichten)")
    if len(faint):
        WARN.append("Comps mit S/N < 100 bei der empfohlenen Belichtung: " + ", ".join(faint["id"]) + " (Handbuch §6.3.1)")
    n_good = int((~comps["id"].isin(set(over["id"]) | set(faint["id"]))).sum())
    if n_good < 4:
        WARN.append(f"nur {n_good} brauchbare Comps im Probebild (mindestens 4 empfohlen)")
AP = aperture_from_fwhm(FWHM_PX) if np.isfinite(FWHM_PX) else {}
if BAYER and np.isfinite(FWHM_RAW) and FWHM_RAW < MIN_RAW_FWHM_BAYER:
    WARN.append(f"FWHM im Rohbild {FWHM_RAW:.1f} px < {MIN_RAW_FWHM_BAYER:.0f} px: Sterne zu scharf für die Bayer-Kanäle (Kanalschwankungen je nach Pixelfarbe unter dem Sternzentrum) — defokussieren (Handbuch §5.5.1).")
if np.isfinite(FWHM_PX) and FWHM_PX > 25:
    WARN.append(f"FWHM {FWHM_PX:.0f} px sehr groß: Sterne evtl. ringförmig (Gauß-Anpassung unsicher), Himmelsring und Nachbarsterne prüfen.")
if len(FW_ALL) >= 2 and np.std(FW_ALL) / np.mean(FW_ALL) > 0.15:
    WARN.append("FWHM schwankt zwischen den Probebildern um > 15 % — Fokus/Seeing/Nachführung instabil?")
print(f"FWHM {FWHM_PX:.2f} px in der ausgewerteten Ebene" + (f" = {FWHM_RAW:.1f} px im Rohbild" if BAYER else ""))
print(f"Aperturen (Pixel der {'Grün-' if BAYER else ''}Ebene): {AP}")
if V_COMP_MIN is not None:
    print(f"Hellster zulässiger Comp bei {T_REC} s: V ≥ {V_COMP_MIN:.2f} (hellere Comps sättigen/überschreiten die Obergrenze) — NB03 übernimmt das über den Plan")
print(f"Längste Belichtung bis zur Obergrenze ({LIMIT:.0f} ADU): {'—' if T_MAX is None else f'{T_MAX:.1f} s'};  empfohlen: {'—' if T_REC is None else f'{T_REC} s'}")
for w in WARN:
    print("⚠", w)

## Schritt 5 — Plan speichern

Im Modus `real` wird der Plan in `PHOTOMETRIE_PY_DATA/plans/` abgelegt. NB04 liest ihn über `VARPHOT_PLAN=<Datei>` und übernimmt die Aperturen; die Methode (Meldefilter, `OBSTYPE`) steht für NB06 bereit. Im Demo-Modus wird nichts gespeichert.

In [ ]:
PLAN = ObservingPlan(target=TARGET, ra_deg=RA_DEG, dec_deg=DEC_DEG, v_mean=V_MEAN, v_max=V_MAX, method=METHOD, aavso_filter=AAVSO_FILTER, obstype=OBSTYPE,
                     camera_model=MODEL, camera_setting=SETTING, bayer_pattern=PATTERN if BAYER else "", site_name=SITE.name if SITE else "",
                     fwhm_px=None if not np.isfinite(FWHM_PX) else round(FWHM_PX, 2), fwhm_raw_px=None if not np.isfinite(FWHM_RAW) else round(FWHM_RAW, 2),
                     aperture=AP, offset_adu=OFFSET, max_adu=MAX_ADU, limit_adu=LIMIT, limit_source=LIMIT_SRC,
                     exptime_max_s=None if T_MAX is None else round(T_MAX, 2), exptime_recommended_s=T_REC, warnings=WARN,
                     v_comp_min=None if V_COMP_MIN is None else round(V_COMP_MIN, 2), moon=MOON,
                     field_checks=[] if FIELD is None else FIELD[["id", "role", "V", "raw_max", "t_max_s", "snr", "ok_snr", "limiting"]].round(3).to_dict("records"),
                     notes=f"{len(frames)} Probebilder, {'Demo' if MODE == 'demo' else TEST_DIR}")
if MODE == "real":
    path = save_plan(PLAN)
    print("gespeichert:", path, "\nin NB04 verwenden:  VARPHOT_PLAN=" + str(path))
else:
    print("DEMO: nichts gespeichert. Plan wäre:"); display(pd.Series({k: v for k, v in PLAN.__dict__.items()}))